# CenterLoRA-ESM2 v1 — development-only screen

This notebook task-adapts ESM-2 35M with LoRA and classifies the central candidate lysine. It runs seed 42 on the protein-grouped development split. It does **not** read the independent test or perform a full-data refit. Select a GPU runtime.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

## Clone or update the private repository

Add a fine-grained read-only GitHub token to Colab Secrets as `GITHUB_TOKEN` and enable notebook access.

In [ ]:
from pathlib import Path
import os
import subprocess

try:
    from google.colab import userdata
    try:
        GITHUB_TOKEN = userdata.get('GITHUB_TOKEN')
    except Exception:
        GITHUB_TOKEN = None
except ImportError:
    GITHUB_TOKEN = None

git_environment = os.environ.copy()
git_environment['GIT_TERMINAL_PROMPT'] = '0'
askpass_path = Path('/tmp/ubipred_git_askpass.sh')
if GITHUB_TOKEN:
    askpass_path.write_text(
        '#!/bin/sh\n'
        'case "$1" in\n'
        '  *Username*) printf "%s\\n" "x-access-token" ;;\n'
        '  *) printf "%s\\n" "$GITHUB_TOKEN" ;;\n'
        'esac\n',
        encoding='utf-8',
    )
    askpass_path.chmod(0o700)
    git_environment['GIT_ASKPASS'] = str(askpass_path)
    git_environment['GITHUB_TOKEN'] = GITHUB_TOKEN

REPO_ROOT = Path('/content/Ubiquitination-Sites-Prediction')
try:
    if not (REPO_ROOT / '.git').exists():
        subprocess.run([
            'git', 'clone', '--recurse-submodules',
            'https://github.com/kaleabnega/Ubiquitination-Sites-Prediction.git',
            str(REPO_ROOT),
        ], check=True, env=git_environment)
    else:
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'pull', '--ff-only'],
            check=True, env=git_environment,
        )
        subprocess.run(
            ['git', '-C', str(REPO_ROOT), 'submodule', 'update', '--init', '--recursive'],
            check=True, env=git_environment,
        )
except subprocess.CalledProcessError as error:
    raise RuntimeError(
        'Git access failed. Add GITHUB_TOKEN to Colab Secrets for a private repository.'
    ) from error
finally:
    askpass_path.unlink(missing_ok=True)

os.chdir(REPO_ROOT)
print('Project:', REPO_ROOT)
subprocess.run(['git', 'rev-parse', 'HEAD'], check=True)

In [ ]:
%pip install -q -r experiment/requirements-colab.txt
%pip uninstall -y torchao

In [ ]:
import json
import platform
import sys
import peft
import sklearn
import torch
import transformers

print('Python:', sys.version)
print('Platform:', platform.platform())
print('scikit-learn:', sklearn.__version__)
print('PyTorch:', torch.__version__)
print('Transformers:', transformers.__version__)
print('PEFT:', peft.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))
else:
    raise RuntimeError('Select a GPU runtime before starting the screen.')

def run_live(command):
    command = [str(part) for part in command]
    print('$', ' '.join(command), flush=True)
    process = subprocess.Popen(
        command,
        stdout=subprocess.PIPE,
        stderr=subprocess.STDOUT,
        text=True,
        bufsize=1,
    )
    if process.stdout is None:
        raise RuntimeError('Could not open subprocess output stream')
    for line in process.stdout:
        print(line, end='', flush=True)
    return_code = process.wait()
    if return_code != 0:
        raise subprocess.CalledProcessError(return_code, command)

## Preflight tests

In [ ]:
run_live([
    sys.executable, '-u', '-m', 'unittest', 'discover',
    '-s', 'experiment/tests',
])

## Run the seed-42 screen

The first download is larger than the previous 8M model and may delay epoch 1. Outputs are written directly to Google Drive and survive a runtime disconnection. If the completed metrics already exist, the cell skips retraining.

In [ ]:
from datetime import date

RUN_DIR = (
    Path('/content/drive/MyDrive/MMUbiPred-research/experiments')
    / f'{date.today().isoformat()}-center-lora-esm2-v1-seed42'
)
RUN_DIR.mkdir(parents=True, exist_ok=True)
METRICS_PATH = RUN_DIR / 'validation_metrics.json'

if METRICS_PATH.exists():
    print('Completed metrics already exist; training was skipped:', METRICS_PATH)
else:
    run_live([
        sys.executable, '-u', 'experiment/scripts/train.py',
        '--config', 'experiment/configs/center_lora_esm2_v1.json',
        '--seed', '42',
        '--output-dir', RUN_DIR,
        '--development-only',
    ])
print('Persistent run directory:', RUN_DIR)

## Inspect the go/no-go result

The stabilized compatible-baseline mean fixed-threshold MCC is 0.5639. A single seed does not establish superiority; it only decides whether a three-seed paired run is warranted.

In [ ]:
metrics = json.loads(METRICS_PATH.read_text())
display(metrics['fixed_threshold'])
display(metrics['validation_selected_threshold'])
display(metrics['development_selection'])
display({
    'branch_names': metrics['branch_names'],
    'mean_component_weights': metrics['mean_component_weights'],
    'prediction_diagnostics': metrics['prediction_diagnostics'],
})

mcc = float(metrics['fixed_threshold']['mcc'])
if mcc >= 0.56:
    decision = 'COMPETITIVE SCREEN: report the output before running more seeds.'
elif mcc >= 0.54:
    decision = 'BORDERLINE SCREEN: inspect the learning curve before deciding.'
else:
    decision = 'STOP THIS VERSION: do not run additional seeds.'
print(decision)
print('Locked independent test accessed:', False)